In [ ]:
# uproot, to run localy

import uproot
import awkward as ak


with uproot.open("../data/train/HToBB_000.root:tree") as tree:  # type: ignore
    #  file.keys() -> ['tree;1']
    for feature in tree.keys():
        print(feature)

part_px
part_py
part_pz
part_energy
part_deta
part_dphi
part_d0val
part_d0err
part_dzval
part_dzerr
part_charge
part_isChargedHadron
part_isNeutralHadron
part_isPhoton
part_isElectron
part_isMuon
label_QCD
label_Hbb
label_Hcc
label_Hgg
label_H4q
label_Hqql
label_Zqq
label_Wqq
label_Tbqq
label_Tbl
jet_pt
jet_eta
jet_phi
jet_energy
jet_nparticles
jet_sdmass
jet_tau1
jet_tau2
jet_tau3
jet_tau4
aux_genpart_eta
aux_genpart_phi
aux_genpart_pid
aux_genpart_pt
aux_truth_match


In [7]:
with uproot.open("../data/train/HToBB_000.root:tree") as tree: # type: ignore
    arr = tree.arrays( # type: ignore
        ["part_px", "part_py", "part_energy", "jet_pt", "jet_nparticles", "label_Hbb", "label_Hgg"],
        library="ak"
    )

print("part_px type:", ak.type(arr["part_px"]))  # total events (jets) * var = variable length * datatype

i = 0
print("jet_pt:", arr["jet_pt"][i])
print("jet_nparticles:", arr["jet_nparticles"][i])
print("label_Hbb:", arr["label_Hbb"][i])
print("label_Hgg:", arr["label_Hgg"][i])

print("First 10 part_px:", arr["part_px"][i][:10])
print("First 10 part_py:", arr["part_py"][i][:10])
print("First 10 part_energy:", arr["part_energy"][i][:10])
print("Constituent (particles) amount:", len(arr["part_px"][i]))

part_px type: 100000 * var * float32
jet_pt: 856.0711
jet_nparticles: 33.0
label_Hbb: True
label_Hgg: False
First 10 part_px: [75, 71.9, 50.2, 44.9, 32.8, 20.3, 15.7, 15.9, 12.3, 10.3]
First 10 part_py: [-138, -131, -92.8, -82.9, -50.4, -35.9, -28.9, -22.6, -19.1, -15.5]
First 10 part_energy: [177, 169, 124, 110, 83.8, 46.4, 38.1, 37.4, 30.6, 25.2]
Constituent (particles) amount: 33


In [10]:
import glob
import numpy as np

train_bb_files = sorted(glob.glob("../data/train/HToBB_*.root"))
train_gg_files = sorted(glob.glob("../data/train/HToGG_*.root"))

print("train Hbb files:", train_bb_files)
print("train Hgg files:", train_gg_files)

branches = [
    "part_px",
    "part_py",
    "part_pz",
    "part_energy",
    "part_deta",
    "part_dphi",
    "part_d0val",
    "part_d0err",
    "part_dzval",
    "part_dzerr",
    "part_charge",
    "part_isChargedHadron",
    "part_isNeutralHadron",
    "part_isPhoton",
    "part_isElectron",
    "part_isMuon",
    "jet_pt",
    "jet_energy",
    "jet_nparticles",
]


def load_one_root(path, branches):
    with uproot.open(path + ":tree") as tree:
        arr = tree.arrays(branches, library="ak")
    return arr

train Hbb files: ['../data/train\\HToBB_000.root', '../data/train\\HToBB_001.root', '../data/train\\HToBB_002.root']
train Hgg files: ['../data/train\\HToGG_000.root', '../data/train\\HToGG_001.root', '../data/train\\HToGG_002.root']


In [11]:
train_bb_list = []

for path in train_bb_files:
    arr = load_one_root(path, branches)
    train_bb_list.append(arr)
    print(f"Loaded {path} -> {len(arr['jet_pt'])} jets")

train_bb = ak.concatenate(train_bb_list, axis=0)
print("Total Hbb train jets:", len(train_bb["jet_pt"]))

train_gg_list = []

for path in train_gg_files:
    arr = load_one_root(path, branches)
    train_gg_list.append(arr)
    print(f"Loaded {path} -> {len(arr['jet_pt'])} jets")

train_gg = ak.concatenate(train_gg_list, axis=0)
print("Total Hgg train jets:", len(train_gg["jet_pt"]))

y_train_bb = np.ones(len(train_bb["jet_pt"]), dtype=np.int64)
y_train_gg = np.zeros(len(train_gg["jet_pt"]), dtype=np.int64)

train_arr = ak.concatenate([train_bb, train_gg], axis=0)
y_train = np.concatenate([y_train_bb, y_train_gg], axis=0)

print("Total train jets:", len(train_arr["jet_pt"]))
print("Total train labels:", len(y_train))
print("Hbb count:", y_train.sum())
print("Hgg count:", len(y_train) - y_train.sum())

Loaded ../data/train\HToBB_000.root -> 100000 jets
Loaded ../data/train\HToBB_001.root -> 100000 jets
Loaded ../data/train\HToBB_002.root -> 100000 jets
Total Hbb train jets: 300000
Loaded ../data/train\HToGG_000.root -> 100000 jets
Loaded ../data/train\HToGG_001.root -> 100000 jets
Loaded ../data/train\HToGG_002.root -> 100000 jets
Total Hgg train jets: 300000
Total train jets: 600000
Total train labels: 600000
Hbb count: 300000
Hgg count: 300000


In [12]:
i = 0

print("jet_pt:", train_arr["jet_pt"][i])
print("jet_nparticles:", train_arr["jet_nparticles"][i])
print("real constituents:", len(train_arr["part_px"][i]))
print("label:", y_train[i])

print("first 5 px:", train_arr["part_px"][i][:5])
print("first 5 energy:", train_arr["part_energy"][i][:5])

jet_pt: 856.0711
jet_nparticles: 33.0
real constituents: 33
label: 1
first 5 px: [75, 71.9, 50.2, 44.9, 32.8]
first 5 energy: [177, 169, 124, 110, 83.8]
